# 14 — Subspace Probe v2: faithfulness read-out + GoalScope viz

Play with the trained **v2 subspace projection** (`o = Wᵀ W h`, KL self-distillation; design: `agent_reports/claude/design/subspace_probe_v2_causal_design.md`). Here we apply it in the **faithfulness** setting — steer the projection into the NL “what are you doing at this line?” probe — and render the interactive qualitative visualization (click tokens to pin read-outs).

Read-out producer: `src.goals.subspace_probe_v2.apply_subspace_scope` (returns the same shape as `apply_goalscope`). Renderer: `src.goals.goalscope_viz`.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
while not os.path.isdir('src') and os.getcwd() != '/':
    os.chdir('..')
sys.path.append('.')
print('cwd:', os.getcwd())

import torch
from src.models import ModelandTokenizer
from src.goals.subspace_probe_v2 import SubspaceProbeV2, apply_subspace_scope
from src.goals.goalscope_viz import show_goalscope_qualitative, goalscope_qualitative_html

cwd: /disk/u/arnab/Codes/Projects/goals


## 1. Load the 32B model + the trained v2 probe

In [2]:
MODEL = 'Qwen/Qwen2.5-Coder-32B-Instruct'
CUDA  = '7'   # set to free GPU(s)
PROBE = 'results/subspace_probe_v2/Qwen2.5-Coder-32B-Instruct/r256__L46__a0.1/probe.pt'
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

mt = ModelandTokenizer(model_key=MODEL, dtype=torch.bfloat16, device_map='auto',
                       attn_implementation='eager')

ckpt = torch.load(PROBE, map_location='cpu', weights_only=False)
probe = SubspaceProbeV2(ckpt['config']); probe.W.data.copy_(ckpt['W'])
probe = probe.to(mt.device).eval()
print(f"probe: r={probe.config.r} read=L{probe.config.read_layer} "
      f"band={probe.config.steer_layers[0]}-{probe.config.steer_layers[-1]} "
      f"alpha={probe.config.alpha} ep={ckpt.get('epoch','?')}")

Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

probe: r=256 read=L46 band=35-46 alpha=0.1 ep=4


## 2. Build a code prompt
Any prompt with a `<code>…</code>` block works (the code segment is auto-located). Edit `CODE` freely.

In [3]:
from src.functional import generate_with_patch
from rich import print as rich_print

msgs = [
    {'role': 'system', 'content': 'You are a helpful assistant proficient in programming. '
        'Always wrap your code inside <code></code> blocks. Just the code, no need to comment or explain.'},
    {'role': 'user', 'content': 'Implement the merge-sort algorithm in python'},
    {'role': 'assistant', 'content': '<code>\n'},
]
prompt = mt.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False,
                                             continue_final_message=True)

gen = generate_with_patch(
    mt=mt,
    inputs=prompt,
    n_gen_per_prompt=1,
    do_sample=False,
    remove_prefix=True,
    max_new_tokens=256
)[0]
rich_print(f"[white]{prompt}[/white][sea_green3]{gen}[/sea_green3]")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


<|im_start|>system
You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just the 
code, no need to comment or explain.<|im_end|>
<|im_start|>user
Implement the merge-sort algorithm in python<|im_end|>
<|im_start|>assistant
<code>
def merge_sort(arr):
    if len(arr) > 1:
        mid = len(arr) // 2
        L = arr[:mid]
        R = arr

        merge_sort(L)
        merge_sort(R)

        i = j = k = 0

        while i < len(L) and j < len(R):
            if L < R:
                arr = L
                i += 1
            else:
                arr = R
                j += 1
            k += 1

        while i < len(L):
            arr = L
            i += 1
            k += 1

        while j < len(R):
            arr = R
            j += 1
            k += 1
</code>

## 3. Apply the subspace projection (faithfulness) at every code token

In [6]:
# o = WᵀW·h at each code token, steered (additive_norm) into the NL faithfulness probe.
# read layer / steer band / alpha come from the probe config.
res = apply_subspace_scope(mt, probe, prompt + gen, mode='causality',
                           max_new_tokens=32, include_noncode=True)
print(sum(1 for r in res if r['scoped']), 'scoped tokens')
for r in [x for x in res if x['scoped']][:8]:
    print(f"  {r['token']!r:>16} -> {r['patched_gen']!r}")

177 scoped tokens
              '\n' -> 'bool check(auto value){ return value > 0; }'
               '<' -> 'bool check(auto value){ return value > 0; }'
            'code' -> 'bool check(auto value){ return value > 0; }'
             '>\n' -> 'bool check(auto value){ return value > 0; }'
             'def' -> 'bool check(auto value){ return value > 10; }'
          ' merge' -> 'bool check(auto value){ return value > 0; }'
           '_sort' -> 'bool check(auto value){ return value > 0; }'
            '(arr' -> 'bool check(auto value){ return value > 0; }'


## 4. Interactive GoalScope visualization
Click a highlighted token to pin its read-out; drag the middle divider to resize. Saved as a standalone `.html` too.

In [5]:
import pathlib; pathlib.Path('notebooks/diff/visualizations').mkdir(parents=True, exist_ok=True)
goalscope_qualitative_html(res, title='stable_marriage — subspace v2 (faithfulness)',
                           save_path='notebooks/diff/visualizations/subspace_v2_faithfulness.html')
show_goalscope_qualitative(res, title='stable_marriage — subspace v2 (faithfulness)', height=820)

/disk/u/arnab/miniconda3/envs/goals/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
